source: https://huggingface.co/spaces/manu02/DINOv3-Interactive-Patch-Cosine-Similarity/tree/main

In [ ]:
# If needed, install interactive backend & widgets (uncomment as needed):
# %pip install -q ipympl ipywidgets

# Enable interactive Matplotlib for clicks & live updates:
# %matplotlib widget

%matplotlib ipympl


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # laadt automatisch .env uit je projectmap

token = os.getenv("HF_TOKEN")

In [ ]:
import numpy as np
from PIL import Image
import torch
from torchvision import transforms
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.patches import Rectangle
from transformers import AutoModel

try:
    import ipywidgets as Widget
except Exception:
    Widget = None  # slider is optional

# ---------- Image I/O ----------
def load_image(path: str) -> Image:
    """Load an image from PATH and return a PIL RGB image."""
    return Image.open(path).convert("RGB")

# ---------- Preprocessing (custom, NO resize) ----------
def crop_to_patchsize(pil_img: Image, patchsize: int=16) -> Image:
    """Pad PIL image on right/bottom so (h,w) are multiples of `patchsize`."""
    w, h = pil_img.size
    h_crop = (h // patchsize) * patchsize
    w_crop = (w // patchsize) * patchsize
    return pil_img.crop((0, 0, w_crop, h_crop))

def np_array(pil_img: Image, dtype: np.dtype =np.uint8) -> np.ndarray:
    return np.array(pil_img, dtype)
    
def preprocess_image(pil_img: Image, patchsize: int, device: torch.device) -> torch.Tensor:
    """Crop (right/bottom) -> ToTensor -> Normalize (ImageNet stats)."""
    img_padded = crop_to_patchsize(pil_img, patchsize)
    transform = transforms.Compose([
        transforms.ToTensor(),  # CxHxW in range [0.0, 1.0]
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    tensor = transform(img_padded).unsqueeze(0).to(device)  # (1,3,H,W)
    return tensor




### Setting Torch Device 

In case it is impossible to perform mass method substitution in tangled code, or if you intend to work with models in the future: Consider using a ***remote machine with a GPU***. 
This is usually what Mac users do, in my experience. You can use Google Colab or other similar services if your model is not very large.

In [ ]:
# setting device to first of these: [GPU, MPS (MacOS), CPU]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

torch.set_default_device(device)

print(f"Using Torch device '{device}'")

### Data loading

In [ ]:
# ---- Image URLs for one or two image mode ----
# path_left = "../data/img/202604-OS-IMGs-Steregushchy-Klasse/532-Boikiy-Sterre-20130726.jpeg"
# label_left = "Boikiy 532 [2013]"
# path_right ="../data/img/202604-OS-IMGs-Steregushchy-Klasse/545-Stoikiy-Sterre-20181124.jpg"
# label_right = "Stoikiy 545 [2018]"

# path_left = "../data/img/202604-OS-IMGs-Steregushchy-Klasse/530-Steregushchiy-Stere-20130219.JPG"
# label_left= "Steregushchy 530 [2013]"
# path_right = "../data/img/202604-OS-IMGs-Steregushchy-Klasse/550-Tsydenzhapov-Sterre-20200622.jpeg"
# label_right= "Tsydenzhapov 550 [2020]"

# path_left = "../data/img/202604-OS-IMGs-Steregushchy-Klasse/532-Boikiy-Sterre-20260222.jpeg"
# label_left = "Soobrazitelny 531 [?]"
# path_right ="../data/img/202604-OS-IMGs-Steregushchy-Klasse/532-Boikiy-Sterre-20260115.jpeg"
# label_right = "Soobrazitelny 531 [?]"

# single image
# path_left = "../data/img/202604-OS-IMGs-Steregushchy-Klasse/531-532-545-Sterre-20181203.jpg"
# label_left = "Soobrazitelny 531 [2011], Boikiy 532 [2013], Boikiy 532 [2013], Stoikiy 545 [2014]"

### CZSK

# all - ok - similar sides: port-beam - port-beam
# works the same as the zoom as there is no rescaling
# downside is that the details are not visible: make zoom
# path_left = "../data/img/202604-CZSK-IMGs-Stere-Klasse/2026-531-Soobrazitelny-Stere/HJQ_2938.JPG"
# label_left = "Soobrazitelny 531 [2026?]"
# path_right ="../data/img/202604-CZSK-IMGs-Stere-Klasse/2026-531-Soobrazitelny-Stere/HJQ_2939.JPG"
# label_right = "Soobrazitelny 531 [2026?]"

### CZSK - zooms

# # mast - excelent - similar sides: port-beam + port-bow
# path_left = "../data/img/202604-CZSK-IMGs-Stere-Klasse/2026-531-Soobrazitelny-Stere/crops/HJQ_2938-mast.1024px.png"
# label_left = "Soobrazitelny 531 [2026?]"
# path_right ="../data/img/202604-CZSK-IMGs-Stere-Klasse/2026-531-Soobrazitelny-Stere/crops/HJQ_2943-mast.1024px.png"
# label_right = "Soobrazitelny 531 [2026?]"

# # bridge - ok-ish - sides: port-bow + starbord-beam
# path_left = "../data/img/202604-CZSK-IMGs-Stere-Klasse/2026-531-Soobrazitelny-Stere/crops/HJQ_2943-bridge.1024px.png"
# label_left = "Soobrazitelny 531 [2026?]"
# path_right ="../data/img/202604-CZSK-IMGs-Stere-Klasse/2026-531-Soobrazitelny-Stere/crops/HJQ_2953-bridge.1024px.png"
# label_right = "Soobrazitelny 531 [2026?]"

# # funnel-deck - excelent - oppisite-sides: port-beam + starbord-beam
# path_left = "../data/img/202604-CZSK-IMGs-Stere-Klasse/2026-531-Soobrazitelny-Stere/crops/HJQ_2938-funnel-deck.1024px.png"
# label_left = "Soobrazitelny 531 [2026?]"
# path_right ="../data/img/202604-CZSK-IMGs-Stere-Klasse/2026-531-Soobrazitelny-Stere/crops/HJQ_2953-funnel-deck.1024px.png"
# label_right = "Soobrazitelny 531 [2026?]"

# # funnel-deck - ok - oppisite-sides: port-beam + starbord-quarter
# path_left = "../data/img/202604-CZSK-IMGs-Stere-Klasse/2026-531-Soobrazitelny-Stere/crops/HJQ_2939-deck-stern.1024px.png"
# label_left = "Soobrazitelny 531 [2026?]"
# path_right ="../data/img/202604-CZSK-IMGs-Stere-Klasse/2026-531-Soobrazitelny-Stere/crops/HJQ_2955-deck-stern.1024px.png"
# label_right = "Soobrazitelny 531 [2026?]"

# # bridge/mast - ? - oppisite-sides: port-beam + bow
path_left = "../images/cropped_images/bridge_w_persons3_2025_CASE_OP_OS_4_STERE_BOIKY.JPG"
label_left = "iets randoms"
path_right ="../images/cropped_images/bridge_w_persons4_2025_CASE_OP_OS_4_STERE_BOIKY.JPG"
label_right = "nog iets randoms"

In [ ]:
from huggingface_hub import login

login(token)

In [ ]:
# ---- Available DINOv3 models on Hugging Face ----
# model_id = "facebook/dinov3-vit7b16-pretrain-lvd1689m"
# model_id = "facebook/dinov3-vits16-pretrain-lvd1689m"
# model_id = "facebook/dinov3-convnext-small-pretrain-lvd1689m"
# model_id = "facebook/dinov3-vitb16-pretrain-lvd1689m"
# model_id = "facebook/dinov3-convnext-base-pretrain-lvd1689m"
# model_id = "facebook/dinov3-vits16plus-pretrain-lvd1689m"
# model_id = "facebook/dinov3-convnext-tiny-pretrain-lvd1689m"
# model_id = "facebook/dinov3-vitl16-pretrain-sat493m"
# model_id = "facebook/dinov3-vitl16-pretrain-lvd1689m"
# model_id = "facebook/dinov3-vith16plus-pretrain-lvd1689m"
# model_id = "facebook/dinov3-convnext-large-pretrain-lvd1689m"
# model_id = "facebook/dinov3-vit7b16-pretrain-sat493m"

# ---- User config ----
show_grid = False
show_overlay = True
annotate_indices = False
overlay_alpha = 0.55
patch_size_override = None  # set to 16 to force; None = read from model if available 


# ---- Model setup ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pretrained_model_name = "facebook/dinov3-vits16-pretrain-lvd1689m"  # Load locally
model = AutoModel.from_pretrained(pretrained_model_name).to(device)
model.eval()
ps = patch_size_override if patch_size_override is not None else getattr(getattr(model, "config", object()), "patch_size", 16)


# class State:
#     # TODO what kind of state is this?


#     def __init__(self, pil_img: Image, model, patchsize: int=16, device=torch.device("cpu")):
#         self.img_tensor = preprocess_image(pil_img, patchsize, device)
#         #    _, _, H, W = img_tensor.shape
#         self.img_h, self.img_w = self.img_tensor.shape[-2:]
#         self.n_patch_rows = self.img_h // patchsize
#         self.n_patch_cols = self.img_w // patchsize
#         self.img_np = np_array(crop_to_patchsize(pil_img, patchsize))  # (H,W,3) for display
#         self.patchsize = patchsize
#         self.device = device
#         self.patch_emb = self.encode(model)
#         self.patch_emb_norm = self.patch_emb.reshape(-1, self.patch_emb.shape[-1]) # (4096, 384)

#     def encode(self, model):
#         with torch.no_grad():
#             y = model(self.img_tensor)
#             hs = y.last_hidden_state.squeeze(0).detach().cpu().numpy()  # (1, 4101, 384) for last hidden state
#         n_patches = self.n_patch_rows * self.n_patch_cols  # 4096
#         patch_emb = hs[-n_patches:, :].reshape(self.n_patch_rows, self.n_patch_cols, -1)  # (64, 64, 384)
#         return patch_emb


# ---------- Row/col <-> idx utilities ----------
def rc_to_idx(r, c, cols): return int(r) * cols + int(c)
def idx_to_rc(i, cols):    return (int(i) // cols, int(i) % cols)


# ---------- Small drawing utilities ----------
def init_grid(ax, rows, cols, ps) -> list[plt.Line2D]:
    grid = []
    for r in range(1, rows):
        line = ax.axhline(r * ps - 0.5, lw=0.8, alpha=0.6, color="white", zorder=3)
        grid.append(line)
    for c in range(1, cols):
        line = ax.axvline(c * ps - 0.5, lw=0.8, alpha=0.6, color="white", zorder=3)
        grid.append(line)
    return grid

def grid_set_visible(grid: list[plt.Line2D], is_visible: bool):
    for line in grid:
        line.set_visible(is_visible)

def overlay_set_visible(ax_img: matplotlib.image.AxesImage, is_visible: bool):
    ax_img.set_visible(is_visible)

def draw_indices(ax, rows, cols, ps):
    for r in range(rows):
        for c in range(cols):
            idx = r * cols + c
            ax.text(c * ps + ps / 2, r * ps + ps / 2, str(idx),
                    ha="center", va="center", fontsize=7,
                    color="white", alpha=0.95, zorder=4)

def build_state(pil_img, model, patchsize, device):

    # for model
    img_tensor = preprocess_image(pil_img, patchsize, device)
    
    # for display
    img_np = np_array(crop_to_patchsize(pil_img, patchsize=16))  # (H,W,3) for display

    _, _, H, W = img_tensor.shape
    rows, cols = H // patchsize, W // patchsize
    with torch.no_grad():
        y = model(img_tensor)
        hs = y.last_hidden_state.squeeze(0).detach().cpu().numpy()  # (1, 4101, 384) for last hidden state
        # (4101, 384)  = ([batch_size], num_patches + 1 + num_registers, hidden_dim) for DINOv3

    n_patches = rows * cols  # 4096
    patch_embs = hs[-n_patches:, :].reshape(rows, cols, -1)  # (64, 64, 384)
    X = patch_embs.reshape(-1, patch_embs.shape[-1]) # (4096, 384)
    Xn = normalize_embedding(X)  # (4096, 384)
    return {
        "img": img_np, 
        "ps": patchsize,
        "h": H, "w": W, 
        "rows": rows, "cols": cols, 
        "X": X, "Xn": Xn,
        "ax": None, "overlay_im": None, "sel_rect": None, "best_rect": None, "grid": None
    }

def normalize_embedding(x: np.ndarray, order_norm: int =2, offset: float = 1e-8) -> np.ndarray:
    """
    Normalize each embedding (row) of the matrix X(,c) (to have unit length) by dividing by norm (default L2).

    Arguments
    ---------
    x
        A numpy matrix of shape (n, m)
    order_norm
        The order of the norm to be used, default 2 (L2-norm)
    offset
        Small offset to prevent division by zero
    
    Return
    ------
    x_normalized
        The normalized (by row) numpy matrix(rows, columns) of X.
    """
    assert len(x.shape) in (1,2), f"Unexpected matrix dimensionality of {len(x.shape)}, should be 1 or 2"
    x_norm = np.linalg.norm(x, ord=order_norm, axis=1, keepdims=True)
    # Divide x by its norm.
    x_normalized = x / (x_norm + offset)
    return x_normalized

# TODO use xarray instead of ndarray?

# TODO rename to make function clearer
def upsample_nearest(x: np.ndarray, h: int, w: int, patchsize: int):
    """Nearest upsample for 2D or 3D arrays with last-dim channels."""
    if x.ndim == 2:
        return x.repeat(patchsize, 0).repeat(patchsize, 1)
    elif x.ndim == 3:
        C = x.shape[-1]
        return x.repeat(patchsize, 0).repeat(patchsize, 1).reshape(h, w, C)
    raise ValueError("Unsupported ndim for upsample")

# ---- Load images ----
image_left = load_image(path_left)
image_right = load_image(path_right)
# image_right = None

# ---- Patch similarity visualization ----
if image_right is not None:
    # Two-image mode
    S = [build_state(image_left, model, ps, device), build_state(image_right, model, ps, device)]
    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(20, 11))
    axs = [ax_left, ax_right]
else:
    # Single-image mode
    S = [build_state(image_left, model, ps, device)]
    fig, ax_left = plt.subplots(1, 1, figsize=(20, 11))
    axs = [ax_left]

cmap = plt.get_cmap("magma")


# init plots
for i, (ax, st) in enumerate(zip(axs, S)):
    st["ax"] = ax
    ax.imshow(st["img"], zorder=0)
    ax.set_axis_off()  
    # grid
    st["grid"] = init_grid(ax, st["rows"], st["cols"], st["ps"])
    grid_set_visible(st["grid"], show_grid)
    # if annotate_indices:
    #     draw_indices(ax, st["rows"], st["cols"], st["ps"])
    
    # overlay
    init_scalar = 0.5 * np.ones((st["rows"], st["cols"]), dtype=np.float32)
    rgba = cmap(init_scalar)
    rgba_up = upsample_nearest(rgba, st["h"], st["w"], st["ps"])
    st["overlay_im"] = ax.imshow(rgba_up, alpha=0.0, zorder=1)
    # selected and best patch: TODO add topN for each image?
    st["sel_rect"]  = Rectangle((0, 0), st["ps"], st["ps"], fill=False, lw=2.0, ec="red",    zorder=5)
    st["best_rect"] = Rectangle((0, 0), st["ps"], st["ps"], fill=False, lw=2.0, ec="yellow", zorder=6)
    ax.add_patch(st["sel_rect"])
    ax.add_patch(st["best_rect"])
    st["best_rect"].set_visible(False)

# init selected and best patch/rect
active_side = 0
current_idx = [ (S[0]["rows"] // 2) * S[0]["cols"] + S[0]["cols"] // 2, ]
if image_right is not None:
    current_idx.append((S[1]["rows"] // 2) * S[1]["cols"] + S[1]["cols"] // 2)

def set_titles(src_i=None, self_stats=None, cross_stats=None):
    if image_right is not None:
        axs[0].set_title(f"LEFT  {label_left} • {S[0]['rows']}x{S[0]['cols']} patches • {'ACTIVE' if active_side==0 else ''}", fontsize=10)
        axs[1].set_title(f"RIGHT {label_right} • {S[1]['rows']}x{S[1]['cols']} patches • {'ACTIVE' if active_side==1 else ''}", fontsize=10)
        if src_i is not None and self_stats is not None and cross_stats is not None:
            src_name = "LEFT" if src_i == 0 else "RIGHT"
            tgt_name = "RIGHT" if src_i == 0 else "LEFT"
            fig.suptitle(
                f"Source: {src_name}  |  Self cos ∈ [{self_stats[0]:.3f},{self_stats[1]:.3f}]  •  "
                f"{tgt_name} cos ∈ [{cross_stats[0]:.3f},{cross_stats[1]:.3f}]  |  "
                f"Controls: click=select • arrows=move • '1'/'2'/'t'=switch side",
                fontsize=11,
            )
        else:
            fig.suptitle("Controls: click=select • arrows=move • '1'/'2'/'t'=switch side", fontsize=11)
    else:
        axs[0].set_title(f"{label_left} • {S[0]['rows']}x{S[0]['cols']} patches", fontsize=10)
        fig.suptitle("Controls: click=select • arrows=move", fontsize=11)

def clamp_idx(i: int, st): 
    # makes sure i is one of the image patches (in range[0, r*c-1])
    return int(np.clip(i, 0, st["rows"] * st["cols"] - 1))

def update_selection_rects():
    for i, st in enumerate(S):
        r, c = idx_to_rc(current_idx[i], st["cols"])
        st["sel_rect"].set_xy((c * st["ps"], r * st["ps"]))
    for i, st in enumerate(S):
        st["sel_rect"].set_visible(image_right is None or i == active_side)

# Images: list of all images in scope for querying
    # - images/preprocessed images/ display images?
# Embedding: list of all images' embeddings, per embedding:
    # - st["X"]: 1D embedding
    # - st["Xn"]: 1D normalized embedding
# User Interaction: which embedding (image + patch) is queried?
    #( - S: states [st1, st2, .. st_n] ?)
    # - current_idx[st_i]: list of n selected/queried index for eacht state (image)
    # - q_idx : queried embedding index
# Cosine-similarity:
    # q : queried embedding
    # qn : normalized queried embedding
    # src["Xn"]: : 1d normalized embeddings of same image
    # cos_self : cosine similarity of qn with all embeddings of that same image
    # cos_cross : cosine similarity of qn with all embeddings of another image
# Drawing: 
    # - grid: st["rows"], st["cols"], st["ps"]
    # - st["img_overlay"]: st["rows"], st["cols"], st["ps"], src["h"], src["w"], cos_map_self (2D version of cos_self) / cos_map_cross, q_idx (making it red)
    # - st["sel_rect"] : current_idx[src_i], st["ps"], st["cols"]  # patch_rect
    # - st["best_rect"] : current_idx[tgt_i], st["ps"], st["cols"]  # patch_rect
    # - ax.set_title:  feedback on st["rows"], st["cols"] and cos_map_self.min/max, cos_map_cross.min/max
    # - fig.suptitle: min(cos_similarity), max(cos_similarity)


def compute_and_show_both_from_src(src_i):
    src = S[src_i]
    q_idx = clamp_idx(current_idx[src_i], src)  # queried embedding idx
    q = src["X"][q_idx]  # queried embedding
    # qn = src["Xn"][q_idx]  # Same? which is faster?
    qn = q / (np.linalg.norm(q) + 1e-8)  # normalized embedding (length = 1.0)  | linalg.norm = length
    cos_self = np.matmul(src["Xn"], qn)  # same as: cos_self = src["Xn"] @ qn

    # Standard cos similiarity comp (different outcome)
    # # Step 1: Compute the dot product
    # dot_product = np.dot(src["X"], q)
    # # Step 2: Compute the magnitudes (or lengths) of the vectors
    # magnitude_A = src["Xn"]  #np.linalg.norm(A)
    # magnitude_B = src["Xn"][q_idx]
    # # Step 3: Calculate cosine similarity
    # cosine_similarity = np.divide(dot_product, np.matmul(magnitude_A, magnitude_B))
    # cos_self = cosine_similarity
    
    cos_map_self = cos_self.reshape(src["rows"], src["cols"])
    disp_self = (cos_map_self - cos_map_self.min()) / (np.ptp(cos_map_self) + 1e-8)  # push cos_map_self to range [0,1]
    rgba_self = cmap(disp_self)  # assign colors to range range [0,1]
    r0, c0 = idx_to_rc(q_idx, src["cols"])  # r and c of queried embedding
    # set queried embedding patch to red with alpha: 1.0
    rgba_self[r0, c0, 0:3] = np.array([1.0, 0.0, 0.0])
    rgba_self[r0, c0, 3]   = 1.0
    src["overlay_im"].set_data(upsample_nearest(rgba_self, src["h"], src["w"], src["ps"]))  # create patches of self-similartiy
    src["overlay_im"].set_alpha(overlay_alpha)
    src["best_rect"].set_visible(False)
    if image_right is not None:
        tgt_i = 1 - src_i
        tgt = S[tgt_i]
        cos_cross = np.matmul(tgt["Xn"], qn)  # same as: cos_cross = tgt["Xn"] @ qn
        cos_map_cross = cos_cross.reshape(tgt["rows"], tgt["cols"])
        disp_cross = (cos_map_cross - cos_map_cross.min()) / (np.ptp(cos_map_cross) + 1e-8)
        rgba_cross = cmap(disp_cross)
        tgt["overlay_im"].set_data(upsample_nearest(rgba_cross, tgt["h"], tgt["w"], tgt["ps"]))
        tgt["overlay_im"].set_alpha(overlay_alpha)
        best = int(np.argmax(cos_cross))  # TODO best4 = np.argpartition(a, -4)[-4:]
        br, bc = idx_to_rc(best, tgt["cols"])
        tgt["best_rect"].set_xy((bc * tgt["ps"], br * tgt["ps"]))
        tgt["best_rect"].set_visible(True)
        S[src_i]["best_rect"].set_visible(False)
        set_titles(src_i, (cos_map_self.min(), cos_map_self.max()),
                          (cos_map_cross.min(), cos_map_cross.max()))
    else:
        set_titles(src_i, (cos_map_self.min(), cos_map_self.max()), None)
    fig.canvas.draw_idle()

def on_click(event):
    global active_side
    if event.inaxes is None or event.xdata is None or event.ydata is None:
        return
    side = 0
    if image_right is not None:
        side = 0 if event.inaxes is axs[0] else (1 if event.inaxes is axs[1] else None)
        if side is None: return
    st = S[side]
    r = int(np.clip(event.ydata // st["ps"], 0, st["rows"] - 1))
    c = int(np.clip(event.xdata // st["ps"], 0, st["cols"] - 1))
    current_idx[side] = rc_to_idx(r, c, st["cols"])
    active_side = side
    update_selection_rects()
    compute_and_show_both_from_src(active_side)

def on_key(event):
    global active_side
    global show_grid
    global show_overlay
    global overlay_alpha

    side = active_side
    if image_right is not None:
        if event.key in ("t", "T"):
            active_side = 1 - active_side
            update_selection_rects()
            compute_and_show_both_from_src(active_side); return
        if event.key == "1":
            active_side = 0; update_selection_rects(); compute_and_show_both_from_src(active_side); return
        if event.key == "2":
            active_side = 1; update_selection_rects(); compute_and_show_both_from_src(active_side); return
    st = S[side]
    r, c = idx_to_rc(current_idx[side], st["cols"])

    if event.key in ("g", "G"):
        show_grid = not show_grid
        for st in S:
            grid_set_visible(st["grid"], show_grid)
    if event.key in ("o", "O"):
        show_overlay = not show_overlay
        for st in S:
            overlay_set_visible(st["overlay_im"], show_overlay)
    elif event.key in ("-", "_"):
        overlay_alpha -= 0.05
        overlay_alpha = max(overlay_alpha, 0.0)
    elif event.key in ("+", "="):
        overlay_alpha += 0.05
        overlay_alpha = min(overlay_alpha, 1.0)
    elif event.key == "left":
        c = max(0, c - 1)
    elif event.key == "right":
        c = min(st["cols"] - 1, c + 1)
    elif event.key == "up":
        r = max(0, r - 1)
    elif event.key == "down":
        r = min(st["rows"] - 1, r + 1)
    else:
        return
    current_idx[side] = rc_to_idx(r, c, st["cols"])
    update_selection_rects()
    compute_and_show_both_from_src(active_side)

update_selection_rects()
set_titles()
compute_and_show_both_from_src(active_side)

fig.canvas.mpl_connect("button_press_event", on_click)
fig.canvas.mpl_connect("key_press_event", on_key)

if image_right is not None:
    print("[two-image] Click to select • arrows move on ACTIVE side • '1'/'2'/'t' switch side")
else:
    print("[single-image] Click to select • arrows move selection")
plt.tight_layout()
plt.show()

In [ ]:
from dataclasses import dataclass

@dataclass
class Embedding:
    """Class for keeping track of properties of an preprocessed image."""
    tensor: torch.Tensor
    

    unit_price: float
    quantity_on_hand: int = 0

    def total_cost(self) -> float:
        return self.unit_price * self.quantity_on_hand
    
    


In [ ]:
def preprocess_image(pil_img: Image, patchsize: int, device: torch.device) -> torch.Tensor:
    """Crop (right/bottom) -> ToTensor -> Normalize (ImageNet stats)."""
    img_padded = crop_to_patchsize(pil_img, patchsize)
    transform = transforms.Compose([
        transforms.ToTensor(),  # CxHxW in range [0.0, 1.0]
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    tensor = transform(img_padded).unsqueeze(0).to(device)  # (1,3,H,W)
    return tensor

In [ ]:
patchsize  = 16

# preprocess image
img_tensor = preprocess_image(image_left, patchsize, device)
print(img_tensor.shape)

# img_tensor properties
b, c, h, w = img_tensor.shape  # batch, channels, height, width
rows, cols = h // patchsize, w // patchsize
n_patches = rows * cols  # 4096

# model output
y = model(img_tensor)

hs = y.last_hidden_state.squeeze(0).detach().cpu().numpy()
print(hs.shape)

# 2d embeddings TODO needed?
patch_embs = hs[-n_patches:, :].reshape(rows, cols, -1)  # (64, 64, 384)
print(hs.shape)

# lin embeddings
X = patch_embs.reshape(-1, patch_embs.shape[-1]) # (4096, 384)
X2 = hs[-n_patches:, :] # (4096, 384)
Xn = normalize_embedding(X)  # (4096, 384)
print(X.shape)
print(X2.shape)
print(Xn.shape)



stop = True